In [41]:
# Tottenham,Newcastle
# Aston Villa,Leicester
# Bournemouth,Everton
# Crystal Palace,Chelsea
# Manchester City,West Ham
# Southampton,Brentford
# Brighton,Arsenal
# Fulham,Ipswich
# Liverpool,Manchester United
# Wolves,Nottingham Forest

In [42]:
import requests
import pandas as pd
import json
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
import os

In [43]:
teams_25_26 = [
    {"name":"Arsenal", "id":"3","shortName":"Arsenal","abbr":"ARS"},
    {"name":"Aston Villa", "id":"7","shortName":"Aston Villa","abbr":"AVL"},
    {"name":"Bournemouth", "id":"91","shortName":"Bournemouth","abbr":"BOU"},
    {"name":"Brentford", "id":"94","shortName":"Brentford","abbr":"BRE"},
    {"name" :"Brighton and Hove Albion","id":"36","shortName":"Brighton","abbr":"BHA"},
    {"name":"Burnley","id":"90","shortName":"Burnley","abbr":"BUR"},
    {"name":"Chelsea","id":"8","shortName":"Chelsea","abbr":"CHE"},
    {"name":"Crystal Palace","id":"31","shortName":"C Palace","abbr":"CRY"},
    {"name":"Everton","id":"11","shortName":"Everton","abbr":"EVE"},
    {"name":"Fulham","id":"54","shortName":"Fulham","abbr":"FUL"},
    {"name":"Leeds United","id":"2","shortName":"Leeds","abbr":"LEE"},
    {"name":"Liverpool","id":"14","shortName":"Liverpool","abbr":"LIV"},
    {"name":"Manchester City","id":"43","shortName":"Man City","abbr":"MCI"},
    {"name":"Manchester United","id":"1","shortName":"Man Utd","abbr":"MUN"},
    {"name":"Newcastle United","id":"4","shortName":"Newcastle","abbr":"NEW"},
    {"name":"Nottingham Forest","id":"17","shortName":"Nott'm Forest","abbr":"NFO"},
    {"name":"Sunderland","id":"56","shortName":"Sunderland","abbr":"SUN"},
    {"name":"Tottenham Hotspur","id":"6","shortName":"Spurs","abbr":"TOT"},
    {"name":"West Ham United","id":"21","shortName":"West Ham","abbr":"WHU"},
    {"name":"Wolverhampton Wanderers","id":"39","shortName":"Wolves","abbr":"WOL"}]

# create team name lookup
team_name_lookup = {team['shortName']: team['name'] for team in teams_25_26}

# use lookup to update names
def update_name(row):
    row['h_team'] = team_name_lookup.get(row['h_team'], row['h_team'])
    row['a_team'] = team_name_lookup.get(row['a_team'], row['a_team'])
    return row

In [44]:
boots_ = requests.get('https://fantasy.premierleague.com/api/bootstrap-static/').json()


In [45]:
gw = 6

fixtures = requests.get(f'https://fantasy.premierleague.com/api/fixtures/?event={gw}').json()
boots_ = requests.get('https://fantasy.premierleague.com/api/bootstrap-static/').json()

fixture_dets =  [{'team_a': fix['team_a'], 'team_h': fix['team_h'], 'h_fdr': fix['team_h_difficulty'], 'a_fdr': fix['team_a_difficulty'] } for fix in fixtures]
fix_df = pd.DataFrame(fixture_dets)

team_ids = [{'team_code': team['code'], 'team_name': team['name'], 'team_id': team['id']} for team in boots_['teams']]
id_df = pd.DataFrame(team_ids)

def add_team_name(row):
    away = row['team_a']
    home = row['team_h']

    home_team_row = id_df[id_df['team_id'] == home]['team_name'].values
    away_team_row = id_df[id_df['team_id'] == away]['team_name'].values

    team_home = home_team_row[0] if home_team_row.size > 0 else 'None'
    team_away = away_team_row[0] if away_team_row.size > 0 else 'None'

    if(team_home == 'Man Utd'):
        team_home = 'Manchester United'
    elif(team_away == 'Man Utd'):
        team_away = 'Manchester United'

    if(team_home == 'Man City'):
        team_home = 'Manchester City'
    elif(team_away == 'Man City'):
        team_away = 'Manchester City'

    if(team_home == 'Wolves'):
        team_home = 'Wolverhampton Wanderers'
    elif(team_away == 'Wolves'):
        team_away = 'Wolverhampton Wanderers'

    if(team_home == "Nott'm Forest"):
        team_home = "Nottingham Forest"
    elif(team_away == "Nott'm Forest"):
        team_away = "Nottingham Forest"

    if(team_home == "Spurs"):
        team_home = "Tottenham"
    elif(team_away == "Spurs"):
        team_away = "Tottenham"

    if(team_home == "Newcastle"):
        team_home = "Newcastle United"
    elif(team_away == "Newcastle"):
        team_away = "Newcastle United"


    return pd.Series([team_home, team_away])


fix_df[['team_home', 'team_away']] = fix_df.apply(add_team_name, axis=1)

fix_df.rename(columns={'team_home': 'h_team', 'team_away': 'a_team'}, inplace=True)

fix_df = fix_df.apply(update_name, axis=1)

fix_df.to_csv(f'./fpl_team_ids_{gw}.csv', index=False)


In [46]:
premLeague = "https://sports.williamhill.com/betting/en-gb/football/competitions/OB_TY295/English-Premier-League/matches/OB_MGMB/Match-Betting"

driver = webdriver.Firefox()
driver.get(premLeague)
matches = driver.find_elements(By.CSS_SELECTOR, "article.sp-o-market--default")[0:10]
details = pd.DataFrame({'h_team':[], 'a_team':[], 'WHH':[], 'WHD':[], 'WHA':[]})

# Get links for the matches
# [[match, link]]
for match in matches:
    # matches_links.append([match.text, match.get_attribute('href')])
    # Extract teams
    teams = match.find_element(By.CSS_SELECTOR, 'main.sp-o-market__title span').text
    odds_els = match.find_elements(By.CSS_SELECTOR, 'section.sp-o-market__buttons .sp-betbutton > span')

    # Extract odds
    # odds = [round(int(btn.text.split('/')[0]) / int(btn.text.split('/')[1]) +1, 2)   for btn in match.find_elements(By.CSS_SELECTOR, 'section.sp-o-market__buttons .sp-betbutton > span')]

    odds = []
    for btn in odds_els:
        if btn.text == 'EVS':
            odds.append(round(2.0, 2))
        else:
            odds.append(round(int(btn.text.split('/')[0]) / int(btn.text.split('/')[1]) +1, 2))
    h_odds = 1/odds[0]
    d_odds = 1/odds[1]
    a_odds = 1/odds[2]

    sum_odd_probs = h_odds + d_odds + a_odds
    match_details = {
        'h_team': [teams.split(' v ')[0]],
        'a_team': [teams.split(' v ')[1]],
        'WHH': [round(h_odds/sum_odd_probs, 3)],
        'WHD': [round(d_odds/sum_odd_probs, 3)],
        'WHA': [round(a_odds/sum_odd_probs, 3)]
        }
    match_dets_df = pd.DataFrame(match_details)
    details = pd.concat([details, match_dets_df], ignore_index=True)
    # print(odds)
    # print({'Home': teams.split(' v ')[0], 'Away': teams.split(' v ')[1], 'WHH': odds[0], 'WHD': odds[1], 'WHA': odds[2]  })


driver.close()


details

,h_team,a_team,WHH,WHD,WHA
0,Brentford,Man Utd,0.284,0.260,0.456
1,Chelsea,Brighton,0.527,0.243,0.231
2,Crystal Palace,Liverpool,0.247,0.251,0.502
3,Leeds,Bournemouth,0.323,0.284,0.393
4,Man City,Burnley,0.794,0.134,0.072
5,Nottingham Forest,Sunderland,0.507,0.273,0.221
6,Tottenham,Wolves,0.636,0.211,0.153
7,Aston Villa,Fulham,0.403,0.299,0.299
8,Newcastle,Arsenal,0.292,0.283,0.425
9,Everton,West Ham,0.511,0.270,0.219


In [47]:
details_ = details.apply(update_name, axis=1)
details_.to_csv(f'./odds_{gw}.csv', index=False)
details_

,h_team,a_team,WHH,WHD,WHA
0,Brentford,Manchester United,0.284,0.260,0.456
1,Chelsea,Brighton and Hove Albion,0.527,0.243,0.231
2,Crystal Palace,Liverpool,0.247,0.251,0.502
3,Leeds United,Bournemouth,0.323,0.284,0.393
4,Manchester City,Burnley,0.794,0.134,0.072
5,Nottingham Forest,Sunderland,0.507,0.273,0.221
6,Tottenham,Wolverhampton Wanderers,0.636,0.211,0.153
7,Aston Villa,Fulham,0.403,0.299,0.299
8,Newcastle United,Arsenal,0.292,0.283,0.425
9,Everton,West Ham United,0.511,0.270,0.219


## Add fdr


In [ ]:

odds_ = pd.read_csv(f'./odds_{gw}.csv')
teams = pd.read_csv(f'./fpl_team_ids_{gw}.csv')


def add_fdr(row):

    team_row = teams[teams['h_team'] == row['h_team']]
    h_fdr = team_row['h_fdr'].values[0] if team_row['h_fdr'].values.size > 0 else 'None'
    a_fdr = team_row['a_fdr'].values[0] if team_row['a_fdr'].values.size > 0 else 'None'

    return pd.Series([h_fdr, a_fdr])

odds_[['h_fdr', 'a_fdr']] = odds_.apply(add_fdr, axis=1)
odds_.to_csv(f'./odds_{gw}.csv', index=False)
odds_

,h_team,a_team,WHH,WHD,WHA,h_fdr,a_fdr
0,Brentford,Manchester United,0.284,0.260,0.456,3,3
1,Chelsea,Brighton and Hove Albion,0.527,0.243,0.231,3,4
2,Crystal Palace,Liverpool,0.247,0.251,0.502,4,3
3,Leeds United,Bournemouth,0.323,0.284,0.393,3,2
4,Manchester City,Burnley,0.794,0.134,0.072,2,4
5,Nottingham Forest,Sunderland,0.507,0.273,0.221,2,3
6,Tottenham,Wolverhampton Wanderers,0.636,0.211,0.153,2,3
7,Aston Villa,Fulham,0.403,0.299,0.299,3,4
8,Newcastle United,Arsenal,0.292,0.283,0.425,4,4
9,Everton,West Ham United,0.511,0.270,0.219,2,3


## Update the team names of the odds


In [ ]:
prev_odds = pd.read_csv(os.path.abspath("./E0 25-26.csv"))

prev_odds = prev_odds.rename(columns={'HomeTeam': 'h_team', 'AwayTeam': 'a_team'})

# use lookup to update names
def update_name(row):
    row['h_team'] = team_name_lookup.get(row['h_team'], row['h_team'])
    row['a_team'] = team_name_lookup.get(row['a_team'], row['a_team'])
    return row

prev_odds = prev_odds.apply(update_name, axis=1)

prev_odds.to_csv("./E0 25-26.csv", index=False)


In [53]:
pd.read_csv('./E0 25-26.csv')

,Div,Date,Time,h_team,a_team,FTHG,FTAG,FTR,HTHG,HTAG,...,B365CAHH,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA,BFECAHH,BFECAHA
0,E0,15/08/2025,20:00,Liverpool,Bournemouth,4,2,H,1,0,...,2.03,1.78,2.07,1.85,2.03,1.88,1.94,1.76,2.14,1.86
1,E0,16/08/2025,12:30,Aston Villa,Newcastle United,0,0,D,0,0,...,2.05,1.80,2.02,1.89,2.06,1.80,1.95,1.74,2.14,1.86
2,E0,16/08/2025,15:00,Brighton and Hove Albion,Fulham,1,1,D,0,0,...,1.83,2.03,1.93,2.00,1.84,2.03,1.80,1.96,1.91,2.08
3,E0,16/08/2025,15:00,Sunderland,West Ham United,3,0,H,0,0,...,1.95,1.90,1.97,1.95,1.95,1.94,1.86,1.78,2.02,1.97
4,E0,16/08/2025,15:00,Tottenham,Burnley,3,0,H,1,0,...,1.98,1.88,1.99,1.93,1.98,1.91,1.88,1.83,2.07,1.92
5,E0,16/08/2025,17:30,Wolverhampton Wanderers,Manchester City,0,4,A,0,2,...,1.98,1.88,2.07,1.86,2.04,1.88,1.91,1.81,2.05,1.94
6,E0,17/08/2025,14:00,Chelsea,Crystal Palace,0,0,D,0,0,...,1.85,2.00,1.74,2.21,1.85,2.03,1.79,1.97,1.88,2.12
7,E0,17/08/2025,14:00,Nottingham Forest,Brentford,3,1,H,3,0,...,1.88,1.98,2.18,1.76,1.91,1.98,1.86,1.89,1.96,2.03
8,E0,17/08/2025,16:30,Man United,Arsenal,0,1,A,0,1,...,1.88,1.98,1.91,2.01,1.93,1.98,1.86,1.89,2.00,1.99
9,E0,18/08/2025,20:00,Leeds United,Everton,1,0,H,0,0,...,2.05,1.80,2.02,1.90,2.05,1.86,1.91,1.78,2.14,1.86


In [50]:
prev_odds = pd.read_csv('./E0 25-26.csv')
gw_odds_29 = pd.read_csv('./odds_29.csv')
gw_odds_30 = pd.read_csv('./odds_30.csv')
# gw_odds_31 = pd.read_csv('./odds_31.csv')
gw_odds_32 = pd.read_csv('./odds_32.csv')
# gw_odds_33 = pd.read_csv('./odds_33.csv')
gw_odds_34 = pd.read_csv('./odds_34.csv')
gw_odds_35 = pd.read_csv('./odds_35.csv')

def add_new_odds(row, odds):
    # Get the odds for the current player from the previous odds DataFrame
    player_odds = odds[(odds['h_team'] == row['h_team']) & (odds['a_team'] == row['a_team'])]
    if not player_odds.empty:
        # If the player is found, add the new odds to the row
        row['WHH'] = player_odds['WHH'].values[0]
        row['WHD'] = player_odds['WHD'].values[0]
        row['WHA'] = player_odds['WHA'].values[0]
    return row

prev_odds = prev_odds.apply(add_new_odds, axis=1, odds=gw_odds_29)
prev_odds = prev_odds.apply(add_new_odds, axis=1, odds=gw_odds_30)
prev_odds = prev_odds.apply(add_new_odds, axis=1, odds=gw_odds_32)
prev_odds = prev_odds.apply(add_new_odds, axis=1, odds=gw_odds_34)
prev_odds = prev_odds.apply(add_new_odds, axis=1, odds=gw_odds_35)

prev_odds[['h_team','a_team','WHH', 'WHD', 'WHA']]


,h_team,a_team,WHH,WHD,WHA
0,Manchester United,Fulham,1.65,4.2,5.00
1,Ipswich,Liverpool,8.50,5.5,1.33
2,Arsenal,Wolverhampton Wanderers,1.18,7.0,17.00
3,Everton,Brighton and Hove Albion,2.60,3.5,2.70
4,Newcastle United,Southampton,1.35,5.5,8.00
...,...,...,...,...,...
375,Newcastle United,Everton,NaN,NaN,NaN
376,Nottingham Forest,Chelsea,NaN,NaN,NaN
377,Southampton,Arsenal,NaN,NaN,NaN
378,Tottenham,Brighton and Hove Albion,NaN,NaN,NaN
